# Faithful-MedRAG — MID EVALUATION (Phase 1)

### Reducing Hallucination in Biomedical Question Answering via Learned Answerability Gating and Claim-Level Repair

**Veeral Patidar** · 22JE1065 · Mentor: Prof. Alakesh Kalita
Department of Mathematics and Computing, IIT (ISM) Dhanbad

---

## What you need to do

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Wait. The last cell downloads a zip of all results.
4. Send that zip back.

That's it. The entire implementation is embedded in this notebook — nothing to
upload, nothing to configure.

> **Recommended first run:** in STEP 4, set `FMRAG_SMOKE = "1"` and Run All.
> That finishes in ~10 minutes and proves the whole pipeline works. Then set it
> back to `"0"` and Run All again for the results you keep.

---

## Phase 1 experiments (proposal objectives 1, 2, 3, 5)

| Experiment | What it does | Timeline milestone |
|---|---|---|
| 1 | Vanilla MedRAG baseline on the real MIRAGE benchmark | August |
| 2 | Faithfulness / hallucination evaluation harness | September |
| 3 | Hybrid retrieval + reranking bake-off (E1–E2) | October |
| 3b | Reranker domain-match diagnostic | October |
| 4 | Grounded generation + claim-level repair (E3), configs A–F | November |

Every model is real: `ncbi/MedCPT-*`, `BAAI/bge-reranker-base`,
`ncbi/MedCPT-Cross-Encoder`, `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`, and
a Qwen2.5 generator. Nothing is simulated or stubbed.

Results are labelled **EXECUTED** (run here), **PUBLISHED** (cited paper table),
or **PENDING** (not run). Nothing is estimated and presented as measured.

## Setup

In [ ]:
# ═══ STEP 1 · Check you have a GPU ════════════════════════════════════════
# If this says "NO GPU", stop and fix it first:
#     Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save
# Without a GPU the notebook still runs, but it falls back to a 1.5B generator
# instead of the 7B model the thesis proposal specifies, and it will be slow.
import subprocess, sys

try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True)
    gpu = out.stdout.strip()
except FileNotFoundError:
    gpu = ""

if gpu:
    print("GPU FOUND :", gpu)
    print("\nGood. This run will use Qwen2.5-7B-Instruct (the proposal's generator).")
else:
    print("NO GPU DETECTED")
    print("\nRuntime -> Change runtime type -> T4 GPU -> Save, then re-run this cell.")
    print("You can continue without one, but the generator drops to 1.5B.")


GPU FOUND : Tesla T4, 15360 MiB

Good. This run will use Qwen2.5-7B-Instruct (the proposal's generator).


In [ ]:
# ═══ STEP 2 · Install dependencies ════════════════════════════════════════
# Takes 1-3 minutes. Only installs what Colab is missing.
import importlib, subprocess, sys

NEEDED = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "rank_bm25": "rank_bm25",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "pptx": "python-pptx",
    "PIL": "pillow",
    "reportlab": "reportlab"
}
missing = [pkg for mod, pkg in NEEDED.items()
           if not importlib.util.find_spec(mod)]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

import torch
if torch.cuda.is_available() and not importlib.util.find_spec("bitsandbytes"):
    print("installing bitsandbytes + accelerate for 4-bit 7B generation")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "bitsandbytes", "accelerate"], check=False)

print("\ndependencies ready")


installing: ['rank_bm25', 'python-pptx', 'reportlab']
installing bitsandbytes + accelerate for 4-bit 7B generation

dependencies ready


In [ ]:
# ═══ STEP 3 · Unpack the project code ════════════════════════════
# The complete Faithful-MedRAG implementation is embedded in this notebook
# as a compressed archive, so there is nothing to upload. This cell writes
# it to disk and imports it.
import base64, io, os, sys, tarfile
from pathlib import Path

ROOT = Path('/content/Faithful_MedRAG')
ROOT.mkdir(parents=True, exist_ok=True)

PAYLOAD = "..."

with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD))) as tar:
    tar.extractall(ROOT)

sys.path.insert(0, str(ROOT))
os.environ['FMRAG_ROOT'] = str(ROOT)

n = sum(1 for _ in ROOT.rglob('*.py'))
print(f'unpacked {n} Python modules to {ROOT}')

from fmrag import config as C
C.ensure_dirs()
print()
print(C.describe())


unpacked 24 Python modules to /content/Faithful_MedRAG



/tmp/ipykernel_721/572364608.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(ROOT)


Faithful-MedRAG configuration
  environment      : colab  (Linux-6.6.159+-x86_64-with-glibc2.39)
  device           : cuda
  generator        : Qwen/Qwen2.5-7B-Instruct  [proposal_7b]
                     Generator specified in the thesis proposal. Needs CUDA + bitsandbytes.
  corpus profile   : subset_textbooks
                     Documented subset: PubMedQA gold abstracts + MedRAG textbook distractors.
  retriever        : ncbi/MedCPT-Query-Encoder + BM25 -> RRF
  reranker         : BAAI/bge-reranker-base
  verifier         : MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  seed             : 42


In [ ]:
# ═══ STEP 4 · Scale settings — the ONLY cell you may want to edit ═════════
# Sensible defaults are already set. Read the comments, change nothing if unsure.
import os, importlib, torch

# Generator. On a GPU this picks Qwen2.5-7B-Instruct (4-bit), which is the
# generator named in the thesis proposal.
os.environ["FMRAG_GEN_PROFILE"] = "proposal_7b" if torch.cuda.is_available() else "local_small"

# Retrieval corpus. 'subset_textbooks' = 43,246 passages (PubMedQA gold abstracts
# + MedRAG textbook distractors). 'full_pubmed' is the proposal's 23.9M-chunk
# corpus and needs ~200GB disk — NOT available on free Colab.
os.environ["FMRAG_CORPUS_PROFILE"] = "subset_textbooks"

# ── SMOKE TEST SWITCH ────────────────────────────────────────────────────
# "1" = tiny samples, finishes in ~10 min. Use this for your FIRST run to
#       confirm everything works end to end.
# "0" = real sample sizes. Use this for the run whose results you keep.
os.environ["FMRAG_SMOKE"] = "0"
# ─────────────────────────────────────────────────────────────────────────

from fmrag import config as C
importlib.reload(C)
C.set_all_seeds()
C.ensure_dirs()

# On a CUDA GPU, larger batches are faster. (Small batches win only on Apple MPS,
# where peak activation memory is the binding constraint.)
if torch.cuda.is_available():
    C.PIPE.nli_batch_size = 64
    C.PIPE.rerank_batch_size = 64
    C.PIPE.embed_batch_size = 256

print(C.describe())
print()
print("SMOKE MODE :", "ON  (tiny samples)" if os.environ["FMRAG_SMOKE"] == "1"
      else "OFF (real sample sizes)")
print("scale      :", C.SCALE)


Faithful-MedRAG configuration
  environment      : colab  (Linux-6.6.159+-x86_64-with-glibc2.39)
  device           : cuda
  generator        : Qwen/Qwen2.5-7B-Instruct  [proposal_7b]
                     Generator specified in the thesis proposal. Needs CUDA + bitsandbytes.
  corpus profile   : subset_textbooks
                     Documented subset: PubMedQA gold abstracts + MedRAG textbook distractors.
  retriever        : ncbi/MedCPT-Query-Encoder + BM25 -> RRF
  reranker         : BAAI/bge-reranker-base
  verifier         : MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  seed             : 42

SMOKE MODE : OFF (real sample sizes)
scale      : ScaleConfig(n_retrieval_eval=1000, n_generation_eval=200, n_gate_total=1000, n_ood_eval=200, n_human_eval=300)


In [ ]:
# ═══ STEP 5 · Download the real datasets ══════════════════════════════════
# MIRAGE benchmark (7,663 questions), PubMedQA (1,000 expert items + their gold
# abstracts), MedRAG textbook chunks. All public, no login needed.
# The assertion fails loudly if the MIRAGE download is incomplete.
import json, logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from fmrag import data as D
report = D.acquire_all()
print(json.dumps(report, indent=2))

assert report["mirage_total"] == 7663, (
    f"MIRAGE should hold exactly 7,663 questions, got {report['mirage_total']}")
print("\nMIRAGE verified: 7,663 questions, matching the published benchmark.")


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/2.62k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

... (34 outputs left out)

chunk/Surgery_Schwartz.jsonl: reconstructing file:   0%|          |  0.00B / 30.1MB            

chunk/Surgery_Schwartz.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/125847 [00:00<?, ? examples/s]

{
  "mirage_counts": {
    "medqa": 1273,
    "medmcqa": 4183,
    "pubmedqa": 500,
    "bioasq": 618,
    "mmlu": 1089
  },
  "mirage_total": 7663,
  "corpus_passages": 43246,
  "corpus_by_source": {
    "pubmedqa": 3334,
    "textbooks": 39912
  },
  "grounded_questions": 1000,
  "corpus_profile": "subset_textbooks"
}

MIRAGE verified: 7,663 questions, matching the published benchmark.


In [ ]:
# ═══ STEP 6 · Build the retrieval index ═══════════════════════════════════
# Embeds 43,246 passages with MedCPT. On a T4 this is a few minutes; it took
# 52 minutes on an Apple M1. Cached afterwards, so re-running is instant.
import time
from fmrag import data as D
from fmrag.retrieval import BM25Retriever, MedCPTRetriever

corpus, grounded = D.build_corpus()
print(f"corpus   : {len(corpus):,} passages")
print(f"grounded : {len(grounded):,} questions carrying gold evidence\n")

t0 = time.time(); BM25Retriever(corpus, "main")
print(f"BM25   ready in {time.time()-t0:.0f}s")
t0 = time.time(); MedCPTRetriever(corpus, "main")
print(f"MedCPT ready in {time.time()-t0:.0f}s")


corpus   : 43,246 passages
grounded : 1,000 questions carrying gold evidence

BM25   ready in 6s


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

... (10 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT ready in 889s


---
## Experiment 2 · Does the faithfulness instrument actually work?

Before measuring faithfulness at scale, check the verifier discriminates. Two
true and two fabricated claims are checked against two real abstracts.

This also demonstrates the thesis premise directly: a metric that reads only the
final answer letter would score this answer **fully correct**.

In [ ]:
from fmrag.data import Passage
from fmrag.metrics import faithfulness_report
from fmrag.retrieval import Scored
from fmrag.verify import NLIVerifier, decompose_claims

answer = ("Metformin is the first-line pharmacologic therapy for type 2 diabetes [1]. "
          "It works by reducing hepatic glucose production [1]. "
          "Metformin also cures type 1 diabetes completely [2]. "
          "The drug was discovered on Mars in 1823 [2].")
evidence = [
    Scored(Passage("d1", "Metformin",
        "Metformin is recommended as the first-line pharmacological treatment for "
        "type 2 diabetes mellitus. It lowers blood glucose primarily by decreasing "
        "hepatic gluconeogenesis."), 9.0, 1),
    Scored(Passage("d2", "Type 1 diabetes",
        "Type 1 diabetes is an autoimmune condition requiring lifelong insulin "
        "replacement. No oral agent cures it."), 8.0, 2),
]

claims = decompose_claims(answer)
NLIVerifier().verify(claims, evidence)
for c in claims:
    print(f"[{c.index}] {c.label:13s} P(entail)={c.p_entail:.3f} "
          f"P(contra)={c.p_contradict:.3f} citation_ok={c.citation_ok}")

fr = faithfulness_report(claims)
print(f"\nfaithfulness       : {fr.faithfulness:.3f}")
print(f"hallucination rate : {fr.hallucination_rate:.3f}")
print(f"citation precision : {fr.citation_precision:.3f}")
print("\nAn accuracy metric reading only the final letter would call this correct.")


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[0] supported     P(entail)=0.990 P(contra)=0.042 citation_ok=True
[1] supported     P(entail)=0.998 P(contra)=0.002 citation_ok=True
[2] neutral       P(entail)=0.000 P(contra)=0.319 citation_ok=False
[3] neutral       P(entail)=0.000 P(contra)=0.293 citation_ok=False

faithfulness       : 0.500
hallucination rate : 0.500
citation precision : 0.500

An accuracy metric reading only the final letter would call this correct.


---
## Experiment 3 · Hybrid retrieval and reranking (E1–E2)

Four arms over identical queries and an identical corpus, so any difference is
attributable to the retrieval stage alone:

- `bm25` — Okapi BM25 lexical retrieval
- `medcpt` — MedCPT dense bi-encoder
- `rrf` — the two fused by reciprocal rank fusion, `score(d) = Σ 1/(60+rank)`
- `rrf_bge` — fused candidates reranked by the BGE cross-encoder

Gold relevance labels come from PubMedQA: each question's gold set is the
abstract it was written from. Gold ids are **targets only** — no arm sees them at
query time.

In [ ]:
from experiments import exp3_retrieval
df_ret = exp3_retrieval.main(n_eval=C.SCALE.n_retrieval_eval, tag="main")
df_ret


EXPERIMENT 3 : hybrid retrieval + reranking bake-off  (E1-E2)
Faithful-MedRAG configuration
  environment      : colab  (Linux-6.6.159+-x86_64-with-glibc2.39)
  device           : cuda
  generator        : Qwen/Qwen2.5-7B-Instruct  [proposal_7b]
                     Generator specified in the thesis proposal. Needs CUDA + bitsandbytes.
  corpus profile   : subset_textbooks
                     Documented subset: PubMedQA gold abstracts + MedRAG textbook distractors.
  retriever        : ncbi/MedCPT-Query-Encoder + BM25 -> RRF
  reranker         : BAAI/bge-reranker-base
  verifier         : MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  seed             : 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


EXPERIMENT 3 RESULTS  (actually executed)
   mode  n_queries  Recall@5  Recall@10  Recall@20    MRR  nDCG@10  mean_total_s
   bm25       1000    0.5525     0.6024     0.6433 0.8885   0.6353        0.2085
 medcpt       1000    0.7078     0.7822     0.8341 0.9435   0.7824        0.0241
    rrf       1000    0.6501     0.7421     0.8202 0.9415   0.7430        0.2701
rrf_bge       1000    0.6355     0.7127     0.7896 0.9208   0.7141        0.9250


,mode,n_queries,Recall@5,Recall@10,Recall@20,MRR,nDCG@5,nDCG@10,nDCG@20,mean_total_s,...,Recall@20_ci95,Hit@20,Hit@20_ci95,nDCG@20_ci95,MRR_ci95,wall_clock_s,mean_lexical_s,mean_dense_s,mean_fuse_s,mean_rerank_s
0,bm25,1000,0.552456,0.602361,0.643296,0.888492,0.616442,0.635255,0.650088,0.208483,...,0.018380,0.963,0.011705,0.016625,0.017407,208.539551,0.208481,NaN,NaN,NaN
1,medcpt,1000,0.707821,0.782181,0.834125,0.943529,0.752496,0.782373,0.801646,0.024082,...,0.013896,0.994,0.004789,0.012972,0.011935,24.188942,NaN,0.024081,NaN,NaN
2,rrf,1000,0.650115,0.742111,0.820209,0.941483,0.705719,0.743037,0.771995,0.270149,...,0.014454,0.993,0.005170,0.013154,0.012486,270.216854,0.247774,0.022224,0.000144,NaN
3,rrf_bge,1000,0.635463,0.712656,0.789649,0.920848,0.682739,0.714140,0.742211,0.925002,...,0.015374,0.992,0.005524,0.013937,0.013920,925.081615,0.233651,0.022567,0.000141,0.653842


---
## Experiment 3b · Why reranking helped or hurt

On an Apple M1 development run over this corpus, **both** RRF fusion and BGE
reranking *degraded* recall relative to MedCPT alone. Two explanations are
possible, and this experiment separates them:

1. **Fusion dilution** — RRF is rank-based and weights both runs equally, so
   fusing a much weaker BM25 run drags down a strong dense run.
2. **Reranker domain mismatch** — `bge-reranker-base` is general-domain, whereas
   MedCPT ships its own biomedical cross-encoder trained on PubMed search logs.

Arms: dense alone, dense + general reranker, dense + biomedical reranker, and
fusion + biomedical reranker.

In [ ]:
from experiments import exp3_retrieval
df_diag = exp3_retrieval.main(
    n_eval=min(300, C.SCALE.n_retrieval_eval),
    modes=["medcpt", "medcpt_bge", "medcpt_ce", "rrf_medcpt_ce"],
    tag="rerank_diag")
df_diag


EXPERIMENT 3 : hybrid retrieval + reranking bake-off  (E1-E2)
Faithful-MedRAG configuration
  environment      : colab  (Linux-6.6.159+-x86_64-with-glibc2.39)
  device           : cuda
  generator        : Qwen/Qwen2.5-7B-Instruct  [proposal_7b]
                     Generator specified in the thesis proposal. Needs CUDA + bitsandbytes.
  corpus profile   : subset_textbooks
                     Documented subset: PubMedQA gold abstracts + MedRAG textbook distractors.
  retriever        : ncbi/MedCPT-Query-Encoder + BM25 -> RRF
  reranker         : BAAI/bge-reranker-base
  verifier         : MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  seed             : 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            


EXPERIMENT 3 RESULTS  (actually executed)
         mode  n_queries  Recall@5  Recall@10  Recall@20    MRR  nDCG@10  mean_total_s
       medcpt        300    0.6817     0.7616     0.8147 0.9283   0.7580        0.0250
   medcpt_bge        300    0.6044     0.6934     0.7876 0.9028   0.6860        0.6916
    medcpt_ce        300    0.7647     0.8192     0.8364 0.9806   0.8312        0.6452
rrf_medcpt_ce        300    0.7607     0.8040     0.8170 0.9798   0.8224        0.8256


,mode,n_queries,Recall@5,Recall@10,Recall@20,MRR,nDCG@5,nDCG@10,nDCG@20,mean_total_s,...,Recall@20_ci95,Hit@20,Hit@20_ci95,nDCG@20_ci95,MRR_ci95,wall_clock_s,mean_dense_s,mean_rerank_s,mean_lexical_s,mean_fuse_s
0,medcpt,300,0.681702,0.761556,0.814734,0.928312,0.726819,0.758016,0.777897,0.025028,...,0.026481,0.993333,0.009224,0.024345,0.023972,7.527471,0.025026,NaN,NaN,NaN
1,medcpt_bge,300,0.604373,0.693361,0.787615,0.902812,0.651196,0.686020,0.720472,0.691632,...,0.027776,0.993333,0.009224,0.024461,0.027395,207.509807,0.022895,0.657990,NaN,NaN
2,medcpt_ce,300,0.764690,0.819206,0.836353,0.980648,0.813107,0.831199,0.837802,0.645218,...,0.025482,0.993333,0.009224,0.021311,0.013781,193.586323,0.023392,0.584105,NaN,NaN
3,rrf_medcpt_ce,300,0.760659,0.804032,0.816980,0.979778,0.809744,0.822374,0.827488,0.825566,...,0.026729,0.990000,0.011278,0.022302,0.014459,247.693131,0.022510,0.574461,0.228447,0.000138


---
## Experiment 1 · Vanilla MedRAG baseline on MIRAGE

The weakest arm by design: dense retrieval, free-form generation, no citation
discipline, no verification, no abstain option. It establishes the number to beat
and supplies the generations whose claims the harness verifies.

Published MedRAG numbers load as a **reference frame**, clearly labelled. They
are not a like-for-like comparison: those runs used GPT-4 / GPT-3.5 / 70B-class
backbones over the full 23.9M-document corpus.

In [ ]:
from experiments import exp1_mirage_baseline
df_base = exp1_mirage_baseline.main(n_eval=C.SCALE.n_generation_eval)
df_base


EXPERIMENT 1 : vanilla MedRAG baseline on MIRAGE
Faithful-MedRAG configuration
  environment      : colab  (Linux-6.6.159+-x86_64-with-glibc2.39)
  device           : cuda
  generator        : Qwen/Qwen2.5-7B-Instruct  [proposal_7b]
                     Generator specified in the thesis proposal. Needs CUDA + bitsandbytes.
  corpus profile   : subset_textbooks
                     Documented subset: PubMedQA gold abstracts + MedRAG textbook distractors.
  retriever        : ncbi/MedCPT-Query-Encoder + BM25 -> RRF
  reranker         : BAAI/bge-reranker-base
  verifier         : MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  seed             : 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (4 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


EXPERIMENT 1 RESULTS  (actually executed)
                     subset published_column  n_evaluated  n_in_full_mirage  accuracy_executed_pct  std_pct evidence
                     bioasq       BioASQ-Y/N           16               618                  81.25     9.76 EXECUTED
                    medmcqa          MedMCQA          109              4183                  50.46     4.79 EXECUTED
                      medqa         MedQA-US           33              1273                  72.73     7.75 EXECUTED
                       mmlu         MMLU-Med           28              1089                  82.14     7.24 EXECUTED
                   pubmedqa        PubMedQA*           13               500                  69.23    12.80 EXECUTED
AVERAGE(macro over subsets)          Average          199              7663                  71.16      NaN EXECUTED

micro accuracy over the whole sample: 62.31%  (n=199)

PUBLISHED reference frame (Xiong et al. 2024, Table 6, MedRAG rows):
             

,subset,published_column,n_evaluated,n_in_full_mirage,accuracy_executed_pct,std_pct,evidence
0,bioasq,BioASQ-Y/N,16,618,81.250000,9.757809,EXECUTED
1,medmcqa,MedMCQA,109,4183,50.458716,4.788930,EXECUTED
2,medqa,MedQA-US,33,1273,72.727273,7.752753,EXECUTED
3,mmlu,MMLU-Med,28,1089,82.142857,7.237888,EXECUTED
4,pubmedqa,PubMedQA*,13,500,69.230769,12.800774,EXECUTED
5,AVERAGE(macro over subsets),Average,199,7663,71.161923,NaN,EXECUTED


---
## Experiment 4 · Grounded generation and claim-level repair (E3)

Configurations A–F, each adding exactly one component to the one above, so every
difference is attributable to a single stage.

| Config | Added component |
|---|---|
| A | vanilla MedRAG (baseline) |
| B | + claim-level faithfulness **measurement** only |
| C | + hybrid RRF retrieval |
| D | + domain-matched cross-encoder reranking |
| E | + grounded strict-citation generation |
| F | + **claim-level surgical repair loop** (Contribution A) |

G and H need the trained answerability gate and belong to the End Evaluation.

**This is the slowest cell** — generation plus NLI verification across six
configurations. Expect 20–60 minutes on a T4 at real sample sizes.

In [ ]:
from experiments import exp_ablation
df_abl = exp_ablation.main(
    n_eval=C.SCALE.n_generation_eval,
    only=["A_vanilla_medrag", "B_plus_faithfulness_eval", "C_plus_hybrid_retrieval",
          "D_plus_reranking", "E_plus_grounded_generation", "F_plus_claim_repair"],
    phase="phase1")
df_abl


ABLATION : configurations A-H  (Experiments 1, 2, 4, 6)
Faithful-MedRAG configuration
  environment      : colab  (Linux-6.6.159+-x86_64-with-glibc2.39)
  device           : cuda
  generator        : Qwen/Qwen2.5-7B-Instruct  [proposal_7b]
                     Generator specified in the thesis proposal. Needs CUDA + bitsandbytes.
  corpus profile   : subset_textbooks
                     Documented subset: PubMedQA gold abstracts + MedRAG textbook distractors.
  retriever        : ncbi/MedCPT-Query-Encoder + BM25 -> RRF
  reranker         : BAAI/bge-reranker-base
  verifier         : MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
  seed             : 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


ABLATION RESULTS  (actually executed)
                    config   n  overall_accuracy  selective_accuracy  coverage  faithfulness_micro  hallucination_rate  n_repaired_claims  mean_t_total_s
          A_vanilla_medrag 200            0.5950              0.5950    1.0000                 NaN                 NaN                  0          6.0129
  B_plus_faithfulness_eval 200            0.5950              0.5950    1.0000              0.5934              0.4066                  0          5.8184
   C_plus_hybrid_retrieval 200            0.6100              0.6100    1.0000              0.6254              0.3746                  0          6.0873
          D_plus_reranking 200            0.6450              0.6450    1.0000              0.6224              0.3776                  0          6.3536
E_plus_grounded_generation 200            0.6450              0.6564    0.9750              0.5179              0.4821                  0          7.0387
       F_plus_claim_repair 200       

,config,n,overall_accuracy,selective_accuracy,coverage,faithfulness_micro,hallucination_rate,citation_precision,n_repaired_claims,n_abstained_claims,...,use_repair,use_gate,n_answers,n_claims,faithfulness_macro,faithfulness_macro_ci95,contradicted_rate,neutral_rate,abstained_claims,claims_per_answer
0,A_vanilla_medrag,200,0.595,0.595000,1.000,NaN,NaN,NaN,0,0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B_plus_faithfulness_eval,200,0.595,0.595000,1.000,0.593426,0.406574,0.742424,0,0,...,False,False,200.0,578.0,0.589250,0.040983,0.048443,0.358131,0.0,2.89
2,C_plus_hybrid_retrieval,200,0.610,0.610000,1.000,0.625430,0.374570,0.743590,0,0,...,False,False,200.0,582.0,0.618833,0.040627,0.036082,0.338488,0.0,2.91
3,D_plus_reranking,200,0.645,0.645000,1.000,0.622449,0.377551,0.703333,0,0,...,False,False,200.0,588.0,0.613000,0.040554,0.039116,0.338435,0.0,2.94
4,E_plus_grounded_generation,200,0.645,0.656410,0.975,0.517921,0.482079,0.491124,0,0,...,False,False,200.0,558.0,0.520000,0.045720,0.053763,0.428315,0.0,2.79
5,F_plus_claim_repair,200,0.645,0.653631,0.895,0.650538,0.000000,0.585799,74,195,...,True,False,200.0,558.0,0.652500,0.043509,0.000000,0.000000,195.0,2.79


### How to read this table

- **A → B** changes nothing about generation; it only starts *measuring*
  faithfulness. Any gap between `overall_accuracy` and `faithfulness_micro` at
  row B is the thesis's central finding.
- **C → D** isolates the reranker's contribution to evidence precision.
- **E** introduces citation discipline and the abstain option.
- **F** adds the repair loop. Watch `n_repaired_claims` against
  `n_abstained_claims`: faithfulness can improve by *fixing* claims or by
  *withdrawing* them, and those are clinically different behaviours.

---
## Export results, workbook, presentation and report

In [ ]:
import pandas as pd
from fmrag import report as R

tables = {}
for csv in sorted((C.PHASE1 / "Results").glob("*.csv")):
    try:
        tables[csv.stem[:31]] = pd.read_csv(csv)
    except Exception as e:
        print("skip", csv.name, e)
for ref in sorted((C.SHARED / "References").glob("*.csv")):
    tables[("published_" + ref.stem)[:31]] = pd.read_csv(ref)

if tables:
    xlsx = R.save_excel(tables, C.PHASE1, "Mid_Evaluation_Results")
    print("workbook:", xlsx.name, f"({len(tables)} data sheets + provenance)")
print("\nsheets:", list(tables))


workbook: Mid_Evaluation_Results.xlsx (10 data sheets + provenance)

sheets: ['ablation_configurations', 'ablation_per_item', 'exp1_mirage_baseline_accuracy', 'exp1_mirage_per_item', 'exp1_published_vs_executed', 'exp3_retrieval_bakeoff_main', 'exp3_retrieval_bakeoff_rerank_d', 'exp3_retrieval_per_query_main', 'exp3_retrieval_per_query_rerank', 'published_published_medrag_tabl']


In [ ]:
# Rebuild the deck and the report from the numbers just produced.
# Slides with no backing CSV render an explicit "NOT YET EXECUTED" panel rather
# than a placeholder number.
import subprocess, sys
for script in ["build_mid_deck.py", "build_reports.py"]:
    print("running", script)
    subprocess.run([sys.executable, str(C.ROOT / script)], cwd=str(C.ROOT))

from pptx import Presentation
deck = C.PHASE1 / "Mid_Evaluation_Presentation.pptx"
if deck.exists():
    print(f"\n{deck.name}: {len(Presentation(deck).slides)} slides")
for f in sorted(C.PHASE1.glob("*.*")):
    print(f"   {f.name}  ({f.stat().st_size/1024:.0f} KB)")


running build_mid_deck.py
running build_reports.py

Mid_Evaluation_Presentation.pptx: 41 slides
   Mid_Evaluation_Presentation.pptx  (500 KB)
   Mid_Evaluation_Report.pdf  (194 KB)


In [ ]:
# ═══ FINAL STEP · Bundle everything and download ══════════════════════════
# Zips every result, graph, table and checkpoint this notebook produced, then
# starts a browser download. Send that single zip back and the presentation and
# report get rebuilt from it.
import shutil, zipfile
from pathlib import Path
from fmrag import config as C

OUT = Path("/content/Faithful_MedRAG_MID_EVAL_RESULTS.zip")
src_dirs = [C.PHASE1, C.SHARED]

with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for d in src_dirs:
        for f in Path(d).rglob("*"):
            if f.is_file() and f.suffix.lower() not in {".npy", ".pkl"}:
                z.write(f, f.relative_to(C.ROOT))
            elif f.is_file() and f.suffix.lower() == ".pkl":
                z.write(f, f.relative_to(C.ROOT))      # gate checkpoint is small

n = len(zipfile.ZipFile(OUT).namelist())
print(f"{OUT.name} : {n} files, {OUT.stat().st_size/1024:.0f} KB\n")
for name in sorted(zipfile.ZipFile(OUT).namelist())[:40]:
    print("  ", name)

try:
    from google.colab import files
    files.download(str(OUT))
    print("\ndownload started — check your browser's downloads")
except Exception as e:
    print(f"\nAuto-download unavailable ({e}).")
    print(f"Use the left sidebar: Files -> {OUT.name} -> Download.")


---
## Done

Send back **`Faithful_MedRAG_MID_EVAL_RESULTS.zip`**.

It contains every results CSV, every graph, the Excel workbook, the provenance
records, and the generated deck and report. The presentation and report will be
finalised from exactly those numbers — each slide and table cites the CSV it
came from, so anything in them can be traced back.

If a cell failed, send the error text too; partial results are still useful and
the zip will contain whatever completed.